In [ ]:
import re
from pathlib import Path
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 14,
    "axes.labelsize": 16,
    "axes.titlesize": 18,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 12,
})

MODEL_LABELS = {
    "vanilla_l1": "Vanilla L1",
    "elastic_net": "Elastic Net",
    "adaptive_lasso": "Adaptive Lasso",
    "adaptive_elastic_net": "AEN",
    "topk_baseline": "TopK",
}
COLORS = {
    'TopK': '#9467bd', 
    'Vanilla L1': '#d62728', 
    'Elastic Net': '#ff7f0e', 
    'Adaptive Lasso': '#1f77b4', 
    'AEN': '#2ca02c'
}
MARKERS = {
    'TopK': 'o',
    'Vanilla L1': 's',
    'Elastic Net': 'D',
    'Adaptive Lasso': '^',
    'AEN': 'v'
}

## Spiked Model

In [ ]:
# Create figures directory
os.makedirs("figures/spiked", exist_ok=True)

# 1. Load and prepare data
df_step = pd.read_csv("spiked_step_by_step_history.csv")

def extract_info(run_name):
    match = re.search(r'rho([\d\.]+)-(.*)_sweep', run_name)
    if match:
        return float(match.group(1)), match.group(2)
    return None, None

df_step[['rho', 'model_family']] = df_step['run_name'].apply(lambda x: pd.Series(extract_info(x)))

df_step["model_label"] = df_step["model_family"].map(MODEL_LABELS).fillna(df_step["model_family"])

# 2. Average over last 10 steps to smooth online estimates
last_10 = df_step.sort_values(['run_name', 'step']).groupby('run_name').tail(10)
numeric_cols = last_10.select_dtypes(include=np.number).columns.tolist()
grouped_runs = last_10.groupby('run_name')[numeric_cols].mean().reset_index()

# Re-attach labels
grouped_runs[['rho', 'model_family']] = grouped_runs['run_name'].apply(lambda x: pd.Series(extract_info(x)))
grouped_runs["model_label"] = grouped_runs["model_family"].map(MODEL_LABELS).fillna(grouped_runs["model_family"])

# 3. TRUNCATE to L0 >= 8 and <= 64 for rho = 0.0 ONLY
filtered_runs = grouped_runs[(grouped_runs['l0_active_features'] >= 7.5) & 
                             (grouped_runs['l0_active_features'] <= 64.5) & 
                             (grouped_runs['rho'] == 0.0)].copy()

# 4. Pareto Frontier Logic
def get_pareto_frontier(Xs, Ys, maxX=False, maxY=True):
    if len(Xs) == 0:
        return np.array([])
    sorted_list = sorted([[Xs[i], Ys[i]] for i in range(len(Xs))], reverse=maxX)
    pareto_front = [sorted_list[0]]
    for pair in sorted_list[1:]:
        if maxY:
            if pair[1] >= pareto_front[-1][1]:
                pareto_front.append(pair)
        else:
            if pair[1] <= pareto_front[-1][1]:
                pareto_front.append(pair)
    return np.array(pareto_front)

# 5. Plotting a 1x2 grid just for rho = 0.0
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plt.rcParams.update({"font.family": "sans-serif", "font.size": 14})

metrics = [
    ("dead_neurons_pct", r"Dead Neurons (%) $\downarrow$", False), 
    ("explained_variance", r"Explained Variance $\uparrow$", True),
]

models_to_plot = ['TopK', 'Vanilla L1', 'Elastic Net', 'Adaptive Lasso', 'AEN']


for col, (metric_col, y_label, maximize_y) in enumerate(metrics):
    ax = axes[col]
    
    for model in models_to_plot:
        sub = filtered_runs[filtered_runs['model_label'] == model].dropna(subset=[metric_col, 'l0_active_features'])
        if sub.empty: continue
            
        # Plot all points as a scatter (transparent)
        ax.scatter(
            sub['l0_active_features'],
            sub[metric_col],
            color=COLORS.get(model),
            alpha=0.3,
            s=40,
            marker=MARKERS.get(model, 'o'),
        )
        
        # Get and plot Pareto frontier (solid line)
        pf = get_pareto_frontier(sub['l0_active_features'].values, sub[metric_col].values, maxX=False, maxY=maximize_y)
        if len(pf) > 0:
            ax.plot(
                pf[:, 0],
                pf[:, 1],
                label=model,
                color=COLORS.get(model),
                linewidth=3,
                alpha=0.9,
                marker=MARKERS.get(model, 'o'),
                markersize=8,
            )
    
    ax.set_xlabel(r"Empirical Sparsity ($L_0$) $\downarrow$", fontsize=15)
    ax.set_ylabel(y_label, fontsize=15)
    ax.set_title(y_label.split('$')[0].strip(), fontweight='bold', fontsize=17)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(10, 50)  # Lock x-axis strictly around the oracle sparsity (s=16)

axes[0].legend(frameon=True, loc='best', fontsize=12)

plt.suptitle(r"Training SAEs on the Spiked Model ($\rho=0.0$)", fontsize=20, fontweight='bold', y=0.95)
plt.tight_layout()